# Boids Flocking Simulation

Interactive exploration of the classic [Boids](https://en.wikipedia.org/wiki/Boids) flocking algorithm,
powered by the **Minkowski ECS** engine.

Three simple steering rules — separation, alignment, cohesion — produce
emergent flocking behaviour from 2,000 independent agents.

## Setup

```bash
cd crates/minkowski-py
pip install maturin
maturin develop --release
pip install polars matplotlib jupyter
```

In [ ]:
import minkowski_py as mk
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

## 1. Create and run the simulation

Spawn 2,000 boids in a 500x500 toroidal world. Step 200 frames with history recording.

In [ ]:
sim = mk.BoidsSim(
    n=2000,
    world_size=500.0,
    separation_radius=25.0,
    alignment_radius=50.0,
    cohesion_radius=50.0,
    max_speed=4.0,
    max_force=0.1,
)

sim.step(200, record=True)
print(f"Simulated {sim.current_frame()} frames, {sim.entity_count()} entities")

## 2. Inspect current state as a Polars DataFrame

In [ ]:
df = sim.to_polars()
print(f"Columns: {df.columns}")
print(f"Shape: {df.shape}")
df.head(10)

## 3. Scatter plot — final positions coloured by speed

In [ ]:
df = sim.to_polars()
x = df["pos_x"].to_numpy()
y = df["pos_y"].to_numpy()
speed = np.sqrt(df["vel_x"].to_numpy()**2 + df["vel_y"].to_numpy()**2)

fig, ax = plt.subplots(figsize=(8, 8))
scatter = ax.scatter(x, y, c=speed, cmap="viridis", s=3, alpha=0.7)
plt.colorbar(scatter, ax=ax, label="Speed")
ax.set_xlim(0, 500)
ax.set_ylim(0, 500)
ax.set_aspect("equal")
ax.set_title(f"Boids — frame {sim.current_frame()}")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

## 4. Velocity quiver plot

Show heading direction and magnitude for each boid.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
vx = df["vel_x"].to_numpy()
vy = df["vel_y"].to_numpy()

ax.quiver(x, y, vx, vy, speed, cmap="plasma", scale=100, width=0.002, alpha=0.8)
ax.set_xlim(0, 500)
ax.set_ylim(0, 500)
ax.set_aspect("equal")
ax.set_title("Boids — velocity field")
plt.tight_layout()
plt.show()

## 5. Trajectory history — analyse with Polars

In [ ]:
history = sim.history_to_polars()
print(f"History shape: {history.shape}")
print(f"Columns: {history.columns}")

# Compute per-frame average speed
avg_speed = (
    history
    .with_columns(
        (pl.col("vel_x")**2 + pl.col("vel_y")**2).sqrt().alias("speed")
    )
    .group_by("frame")
    .agg(pl.col("speed").mean().alias("avg_speed"))
    .sort("frame")
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(avg_speed["frame"].to_numpy(), avg_speed["avg_speed"].to_numpy(), linewidth=1)
ax.set_xlabel("Frame")
ax.set_ylabel("Average Speed")
ax.set_title("Boids — average flock speed over time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Parameter sweep — how does separation radius affect flock cohesion?

Run multiple simulations with different separation radii and compare.

In [ ]:
radii = [10.0, 25.0, 50.0, 75.0]
results = []

for r in radii:
    s = mk.BoidsSim(n=1000, separation_radius=r)
    s.step(100, record=True)
    h = s.history_to_polars()
    avg = (
        h.with_columns(
            (pl.col("vel_x")**2 + pl.col("vel_y")**2).sqrt().alias("speed")
        )
        .group_by("frame")
        .agg(pl.col("speed").mean().alias("avg_speed"))
        .sort("frame")
        .with_columns(pl.lit(r).alias("sep_radius"))
    )
    results.append(avg)

combined = pl.concat(results)

fig, ax = plt.subplots(figsize=(10, 4))
for r in radii:
    subset = combined.filter(pl.col("sep_radius") == r)
    ax.plot(
        subset["frame"].to_numpy(),
        subset["avg_speed"].to_numpy(),
        label=f"sep_r={r}",
        linewidth=1,
    )
ax.set_xlabel("Frame")
ax.set_ylabel("Average Speed")
ax.set_title("Effect of separation radius on flock dynamics")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()